<a href="https://colab.research.google.com/github/Juliodelemos/sugarcane-variety-classification-yolov8/blob/main/Cropping_4_varieties_polygn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Mount Google Drive (if necessary to save outputs)
from google.colab import drive
import os

drive.mount('/content/drive')  # Optional: Mount Google Drive for file persistence

# Root directory for the experiment
# IMPORTANT: Insert here the path to the root directory for your experiment, e.g., './my_project_root'
# The example below uses a relative path assuming the notebook is run from the project root.
os.chdir('./')
# List contents of the current directory (optional, uncomment if needed for debugging)
# !ls -F

In [ ]:
import os

# IMPORTANT: Insert here the path to the source directory containing images to be processed.
# Example: './Dataset_Original/colmos_dropped'
src_dir = "./data/input_images_dropped"

# IMPORTANT: Insert here the path to the destination directory where processed images will be saved.
# Example: './Dataset_Processed/nunca_vistas_crop'
dst_dir = "./data/output_never_seen_cropped"

# Check if the source directory exists
if not os.path.exists(src_dir):
    print(f"The directory {src_dir} does not exist.")
else:
    print(f"The directory {src_dir} exists.")

## Image Cropping via Rectangular Bounding Box

### Crop an Entire Directory

In [ ]:
import cv2
import numpy as np
import os

def cortar_objetos_de_interesse(imagem_path, saida_dir, threshold_value=200, min_area=130000, sufixo="cropped"):
    print(f"Loading image: {imagem_path}")

    # Load the image
    imagem = cv2.imread(imagem_path)
    if imagem is None:
        print(f"Image not found or invalid path: {imagem_path}")
        return

    # Create the output directory if it doesn't exist
    os.makedirs(saida_dir, exist_ok=True)

    # Add a 50-pixel black border on all sides of the image
    borda = 50
    imagem_com_borda = cv2.copyMakeBorder(imagem, borda, borda, borda, borda, cv2.BORDER_CONSTANT, value=[0, 0, 0])
    print(f"Black border of {borda}px added on all sides of the image.")

    # Convert the image to grayscale
    gray = cv2.cvtColor(imagem_com_borda, cv2.COLOR_BGR2GRAY)
    print("Image converted to grayscale.")

    # Apply blurring to reduce noise
    gray = cv2.GaussianBlur(gray, (5, 5), 0)
    print("Smoothing applied.")

    # Ignore the black border during binarization
    gray_sem_borda = gray[borda:-borda, borda:-borda]
    _, thresh_sem_borda = cv2.threshold(gray_sem_borda, threshold_value, 255, cv2.THRESH_BINARY)

    # Add the border back to the binarized image
    thresh = cv2.copyMakeBorder(thresh_sem_borda, borda, borda, borda, borda, cv2.BORDER_CONSTANT, value=0)
    print(f"Threshold applied with value: {threshold_value}")

    # Apply morphological operations to remove noise
    kernel = np.ones((5, 5), np.uint8)
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel, iterations=2)
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel, iterations=2)
    print("Morphological operations applied.")

    # Find contours in the binarized image
    contornos, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    print(f"{len(contornos)} contour(s) found.")

    # Check if any contours were detected
    if not contornos:
        print(f"No object of interest found in the image: {imagem_path}")
        return

    # Extract the original file name
    nome_arquivo_original = os.path.basename(imagem_path)
    nome_base, extensao = os.path.splitext(nome_arquivo_original)
    print(f"Original file name: {nome_arquivo_original}")

    # Iterate over contours and crop each object of interest
    contador = 0
    for i, contorno in enumerate(contornos):
        # Get the bounding box coordinates
        x, y, w, h = cv2.boundingRect(contorno)

        # Filter contours based on object characteristics:
        # - Width occupying at least 50% of the image width WITH BORDER
        if w < 0.5 * imagem_com_borda.shape[1]:
            print(f"Contour {i + 1} ignored because its width is too small ({w}).")
            continue

        # Increment the counter
        contador += 1
        print(f"Object {contador}: Bounding box coordinates: x={x}, y={y}, width={w}, height={h}")

        # Crop the Region of Interest (ROI) from the original image
        roi = imagem[y:y+h, x:x+w]

        # Save the cropped image with formatted name: original_name_suffix_counter.extension
        nome_saida = f"{nome_base}_{sufixo}_{contador}{extensao}"
        caminho_saida = os.path.join(saida_dir, nome_saida)
        cv2.imwrite(caminho_saida, roi)
        print(f"Object {contador} saved to: {caminho_saida}")


# Function to process all images in a directory
def processar_diretorio(diretorio_entrada, diretorio_saida, threshold_value=85, min_area=130000, sufixo="cropped"):
    # List all files in the input directory
    arquivos = [f for f in os.listdir(diretorio_entrada) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    print(f"Found {len(arquivos)} images in directory: {diretorio_entrada}")

    # Process each image
    for arquivo in arquivos:
        caminho_imagem = os.path.join(diretorio_entrada, arquivo)
        cortar_objetos_de_interesse(caminho_imagem, diretorio_saida, threshold_value, min_area, sufixo)

# Example usage
# IMPORTANT: Insert here the path to your input images directory.
# Example: './Dataset_Original/colmos'
diretorio_entrada = "./data/input_colmos_rect_crop"

# IMPORTANT: Insert here the path to your output directory for cropped images.
# Example: './Dataset_Processed/colmos_cropped'
diretorio_saida = "./data/output_colmos_cropped"

# Adjust the threshold value as needed
threshold_value = 150  # Experiment with different values here
min_area = 200000  # Minimum area to consider a contour as an object of interest
sufixo = "v03"  # Word to be added to the name of the cropped image

processar_diretorio(diretorio_entrada, diretorio_saida, threshold_value, min_area, sufixo)

### Crop a Single Image

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

def cortar_objetos_de_interesse(imagem_path, saida_dir, threshold_value=85, min_area=130000):
    print(f"Loading image: {imagem_path}")

    # Load the image
    imagem = cv2.imread(imagem_path)
    if imagem is None:
        raise ValueError("Image not found or invalid path.")

    # Create the output directory if it doesn't exist
    os.makedirs(saida_dir, exist_ok=True)

    # Add a 50-pixel black border on all sides of the image
    borda = 50
    imagem_com_borda = cv2.copyMakeBorder(imagem, borda, borda, borda, borda, cv2.BORDER_CONSTANT, value=[0, 0, 0])
    print(f"Black border of {borda}px added on all sides of the image.")

    # Show the image with the black border
    plt.figure(figsize=(8, 8))
    plt.title("Original Image with Black Border")
    plt.imshow(cv2.cvtColor(imagem_com_borda, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.show()

    # Convert the image to grayscale
    gray = cv2.cvtColor(imagem_com_borda, cv2.COLOR_BGR2GRAY)
    print("Image converted to grayscale.")

    # Apply blurring to reduce noise
    gray = cv2.GaussianBlur(gray, (5, 5), 0)
    print("Smoothing applied.")

    # Ignore the black border during binarization
    gray_sem_borda = gray[borda:-borda, borda:-borda]
    _, thresh_sem_borda = cv2.threshold(gray_sem_borda, threshold_value, 255, cv2.THRESH_BINARY)

    # Add the border back to the binarized image
    thresh = cv2.copyMakeBorder(thresh_sem_borda, borda, borda, borda, borda, cv2.BORDER_CONSTANT, value=0)
    print(f"Threshold applied with value: {threshold_value}")

    # Visualize the binarized image
    plt.figure(figsize=(12, 6))
    plt.subplot(1, 2, 1)
    plt.title("Original Image with Black Border")
    plt.imshow(cv2.cvtColor(imagem_com_borda, cv2.COLOR_BGR2RGB))
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.title("Binarized Image")
    plt.imshow(thresh, cmap='gray')
    plt.axis('off')
    plt.show()

    # Apply morphological operations to remove noise
    kernel = np.ones((5, 5), np.uint8)
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel, iterations=2)
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel, iterations=2)
    print("Morphological operations applied.")

    # Find contours in the binarized image
    contornos, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    print(f"{len(contornos)} contour(s) found.")

    # Check if any contours were detected
    if not contornos:
        raise ValueError("No object of interest found in the image.")

    # Draw bounding rectangles on the original image
    imagem_com_retangulos = imagem_com_borda.copy()
    contador = 0
    for i, contorno in enumerate(contornos):
        # Get the bounding box coordinates
        x, y, w, h = cv2.boundingRect(contorno)

        # Filter contours based on object characteristics:
        # - Width occupying at least 90% of the image width WITH BORDER
        print(f"0.9 * imagem_com_borda.shape[1]= {0.9 * imagem_com_borda.shape[1]}.")
        if w < 0.9 * imagem_com_borda.shape[1]:
          print(f"Contour {i + 1} ignored because its width is too small ({w}) 0.9 * imagem_com_borda.shape[1]= {0.9 * imagem_com_borda.shape[1]}.")
          continue

        # Increment the counter
        contador += 1
        print(f"Object {contador}: Bounding box coordinates: x={x}, y={y}, width={w}, height={h}")
        # Draw the bounding rectangle
        cv2.rectangle(imagem_com_retangulos, (x, y), (x + w, y + h), (0, 255, 0), 2)
        cv2.putText(imagem_com_retangulos, f"Area: {w * h}", (x, y - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)

    # Show the original image with bounding rectangles
    plt.figure(figsize=(8, 8))
    plt.title("Original Image with Bounding Rectangles")
    plt.imshow(cv2.cvtColor(imagem_com_retangulos, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.show()

# Example usage
# IMPORTANT: Insert here the path to your input image.
# Example: './Dataset_Original/test/IMG_2615.JPG'
imagem_entrada = "./data/test_single_image/IMG_2615.JPG"

# IMPORTANT: Insert here the path to your output directory for cropped images.
# Example: './Dataset_Processed/test/cropped'
saida_diretorio = "./data/test_single_image/cropped"

# Adjust the threshold value as needed
threshold_value = 150  # Experiment with different values here
min_area = 200000  # Minimum area to consider a contour as an object of interest

cortar_objetos_de_interesse(imagem_entrada, saida_diretorio, threshold_value, min_area)

## Polygon Cropping

### Crop a Single Image (Polygon)

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

# Function to crop objects of interest using a rectangular bounding box (returns only the largest ROI)
def cortar_objetos_de_interesse(imagem_path, saida_dir, threshold_value=85, min_area=130000):
    print(f"Loading image: {imagem_path}")

    # Load the image
    imagem = cv2.imread(imagem_path)
    if imagem is None:
        raise ValueError("Image not found or invalid path.")

    # Create the output directory if it doesn't exist
    os.makedirs(saida_dir, exist_ok=True)

    # Convert the image to grayscale
    gray = cv2.cvtColor(imagem, cv2.COLOR_BGR2GRAY)
    print("Image converted to grayscale.")

    # Apply blurring to reduce noise
    gray = cv2.GaussianBlur(gray, (5, 5), 0)
    print("Smoothing applied.")

    # Binarize the image
    _, thresh = cv2.threshold(gray, threshold_value, 255, cv2.THRESH_BINARY)
    print(f"Threshold applied with value: {threshold_value}")

    # Find contours in the binarized image
    contornos, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    print(f"{len(contornos)} contour(s) found.")

    # Filter contours based on minimum area
    contornos_filtrados = [contorno for contorno in contornos if cv2.contourArea(contorno) >= min_area]
    print(f"{len(contornos_filtrados)} contour(s) filtered with minimum area of {min_area}.")

    if not contornos_filtrados:
        raise ValueError("No contour meets the minimum area requirement.")

    # Find the contour with the largest area
    maior_contorno = max(contornos_filtrados, key=cv2.contourArea)
    print("Contour with the largest area selected.")

    # Get the bounding box coordinates of the largest contour
    x, y, w, h = cv2.boundingRect(maior_contorno)

    # Crop the Region of Interest (ROI)
    roi = imagem[y:y+h, x:x+w]

    # Save the cropped image
    nome_saida = os.path.join(saida_dir, f"object_largest.jpg")
    cv2.imwrite(nome_saida, roi)
    print(f"Object with the largest area saved to: {nome_saida}")

    # Return the ROI
    return [roi]


# Function to crop objects of interest using a polygonal mask (returns only the largest ROI)
def cortar_objetos_de_interesse_poligonal(imagem, threshold_value=85, min_area=130000):
    print("Processing ROI with polygonal mask...")

    # Convert the image to grayscale
    gray = cv2.cvtColor(imagem, cv2.COLOR_BGR2GRAY)
    print("Image converted to grayscale.")

    # Apply blurring to reduce noise
    gray = cv2.GaussianBlur(gray, (5, 5), 0)
    print("Smoothing applied.")

    # Binarize the image
    _, thresh = cv2.threshold(gray, threshold_value, 255, cv2.THRESH_BINARY)
    print(f"Threshold applied with value: {threshold_value}")

    # Find contours in the binarized image
    contornos, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    print(f"{len(contornos)} contour(s) found.")

    # Filter contours based on minimum area
    contornos_filtrados = [contorno for contorno in contornos if cv2.contourArea(contorno) >= min_area]
    print(f"{len(contornos_filtrados)} contour(s) filtered with minimum area of {min_area}.")

    if not contornos_filtrados:
        raise ValueError("No contour meets the minimum area requirement.")

    # Find the contour with the largest area
    maior_contorno = max(contornos_filtrados, key=cv2.contourArea)
    print("Contour with the largest area selected.")

    # Approximate the contour with a polygon (adjust epsilon for higher precision)
    epsilon = 0.001 * cv2.arcLength(maior_contorno, True)  # Value adjusted based on observation
    poligono = cv2.approxPolyDP(maior_contorno, epsilon, True)

    # Create a binary mask
    mascara = np.zeros_like(gray)
    cv2.fillPoly(mascara, [poligono], 255)

    # Apply the mask to the original image
    roi = cv2.bitwise_and(imagem, imagem, mask=mascara)

    # Crop the Region of Interest (ROI) using the polygon's bounding box
    x, y, w, h = cv2.boundingRect(poligono)
    roi_recortada = roi[y:y+h, x:x+w]

    # Return the ROI
    return [roi_recortada]


# Function to crop only the sugarcane and save with transparency (.png)
def recortar_cana_de_acucar_com_transparencia(imagens, saida_dir):
    print("Starting final cropping of sugarcane with transparency...")

    # List to store the cropped images
    imagens_recortadas = []

    # Iterate over the refined images (with black background)
    for i, imagem in enumerate(imagens):
        # Create a binary mask: black pixels (background) will be transparent
        # A pixel is considered black if all channels (B, G, R) are zero
        mascara = np.all(imagem == [0, 0, 0], axis=-1).astype(np.uint8) * 255

        # Invert the mask to set black pixels as transparent
        mascara_invertida = cv2.bitwise_not(mascara)

        # Add an alpha channel (transparency) to the original image
        rgba = cv2.cvtColor(imagem, cv2.COLOR_BGR2BGRA)

        # Apply the inverted mask to the alpha channel
        rgba[:, :, 3] = mascara_invertida

        # Generate the file name with the suffix "_crop" and ".png" extension
        nome_saida = os.path.join(saida_dir, f"object_{i + 1}_v03_crop.png")

        # Save the image with transparency in PNG format
        cv2.imwrite(nome_saida, rgba)
        print(f"Image with transparency saved to: {nome_saida}")

        # Add the ROI to the list
        imagens_recortadas.append(rgba)

    # Return the list of cropped images
    return imagens_recortadas


# Example usage
# IMPORTANT: Insert here the path to your input image.
# Example: './Dataset_Original/data_cropped/CTC_1007/IMG_C_111_1.JPG'
imagem_entrada = "./data/single_image_polygon_crop/IMG_C_111_1.JPG"

# IMPORTANT: Insert here the path to your output directory for processed PNG images.
# Example: './Dataset_Processed/data_png'
saida_diretorio = "./data/output_png_single_image"

# Adjust the threshold value as needed
threshold_value = 15  # Experiment with different values here
min_area = 8500  # Minimum area to consider a contour as an object of interest

# Execute the initial function to get the rectangular ROI with the largest area
imagens_recortadas = cortar_objetos_de_interesse(imagem_entrada, saida_diretorio, threshold_value, min_area)

# Process the ROI with the polygonal function (with epsilon adjustment)
imagens_recortadas_poligonais = []
for i, roi in enumerate(imagens_recortadas):
    print(f"Processing ROI {i + 1} with polygonal mask...")
    rois_poligonais = cortar_objetos_de_interesse_poligonal(roi, threshold_value, min_area)
    imagens_recortadas_poligonais.extend(rois_poligonais)

# Crop only the sugarcane and save with transparency (.png)
imagens_finalmente_recortadas = recortar_cana_de_acucar_com_transparencia(imagens_recortadas_poligonais, saida_diretorio)

# Display the final images using matplotlib
for i, roi in enumerate(imagens_finalmente_recortadas):
    plt.figure()
    plt.title(f"Final ROI {i + 1}")
    plt.imshow(cv2.cvtColor(roi, cv2.COLOR_BGRA2RGBA))  # Use RGBA to display transparency
    plt.axis('off')
    plt.show()

### Batch Polygon Cropping

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

# Function to crop objects of interest using a rectangular bounding box (returns only the largest ROI)
def cortar_objetos_de_interesse(imagem, threshold_value=85, min_area=130000):
    # Convert the image to grayscale
    gray = cv2.cvtColor(imagem, cv2.COLOR_BGR2GRAY)
    # Apply smoothing
    gray = cv2.GaussianBlur(gray, (5, 5), 0)
    # Binarize
    _, thresh = cv2.threshold(gray, threshold_value, 255, cv2.THRESH_BINARY)
    # Find contours
    contornos, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    # Filter by minimum area
    contornos_filtrados = [cnt for cnt in contornos if cv2.contourArea(cnt) >= min_area]
    if not contornos_filtrados:
        return None
    # Select the largest contour
    maior_contorno = max(contornos_filtrados, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(maior_contorno)
    return imagem[y:y+h, x:x+w]

# Function to crop objects of interest using a polygonal mask (returns only the largest ROI)
def cortar_objetos_de_interesse_poligonal(imagem, threshold_value=85, min_area=130000):
    # Convert to grayscale
    gray = cv2.cvtColor(imagem, cv2.COLOR_BGR2GRAY)
    # Smooth
    gray = cv2.GaussianBlur(gray, (5, 5), 0)
    # Binarize
    _, thresh = cv2.threshold(gray, threshold_value, 255, cv2.THRESH_BINARY)
    # Find contours
    contornos, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    # Filter by minimum area
    contornos_filtrados = [cnt for cnt in contornos if cv2.contourArea(cnt) >= min_area]
    if not contornos_filtrados:
        return None
    # Select the largest contour
    maior_contorno = max(contornos_filtrados, key=cv2.contourArea)
    # Approximate the polygon
    epsilon = 0.001 * cv2.arcLength(maior_contorno, True)
    poligono = cv2.approxPolyDP(maior_contorno, epsilon, True)
    # Create mask
    mascara = np.zeros_like(gray)
    cv2.fillPoly(mascara, [poligono], 255)
    # Apply mask
    roi = cv2.bitwise_and(imagem, imagem, mask=mascara)
    # Crop with bounding box
    x, y, w, h = cv2.boundingRect(poligono)
    return roi[y:y+h, x:x+w]

# Function to remove background and save as PNG
def remover_fundo_e_salvar(imagem, saida_path):
    # Create transparency mask
    mascara = np.all(imagem == [0, 0, 0], axis=-1).astype(np.uint8) * 255
    mascara_invertida = cv2.bitwise_not(mascara)
    # Add alpha channel
    rgba = cv2.cvtColor(imagem, cv2.COLOR_BGR2BGRA)
    rgba[:, :, 3] = mascara_invertida
    # Save
    cv2.imwrite(saida_path, rgba)

In [ ]:
# Directories
# IMPORTANT: Insert here the path to your input images directory.
# Example: './Dataset_Original/VERTIX12-9-MESES-AMBIENTE-A-PLANTA/colmos'
diretorio_origem = "./data/input_colmos_polygon_batch"

# IMPORTANT: Insert here the path to your output directory for processed images.
# Example: './Dataset_Processed/VERTIX12-9-MESES-AMBIENTE-A-PLANTA/colmos/cropped'
diretorio_destino = "./data/output_colmos_polygon_batch_cropped"

# Parameters
threshold_value = 150
min_area = 130000

# Create destination directory
os.makedirs(diretorio_destino, exist_ok=True)

# Process all images in the source directory
for nome_arquivo in os.listdir(diretorio_origem):
    if nome_arquivo.lower().endswith(('.png', '.jpg', '.jpeg')):
        # Full path of the image
        imagem_path = os.path.join(diretorio_origem, nome_arquivo)
        print(f"\nProcessing: {imagem_path}")

        # Load image
        imagem = cv2.imread(imagem_path)
        if imagem is None:
            print(f"Error loading: {imagem_path}")
            continue

        # Processing pipeline
        try:
            # 1. Crop rectangular ROI (largest area)
            roi_retangular = cortar_objetos_de_interesse(imagem, threshold_value, min_area)
            if roi_retangular is None:
                print("No object found in the rectangular ROI.")
                continue

            # 2. Crop polygonal ROI
            roi_poligonal = cortar_objetos_de_interesse_poligonal(roi_retangular, threshold_value, min_area)
            if roi_poligonal is None:
                print("No object found in the polygonal ROI.")
                continue

            # 3. Remove background and save as PNG
            nome_saida = os.path.splitext(nome_arquivo)[0] + "_v12_crop.png"
            caminho_saida = os.path.join(diretorio_destino, nome_saida)
            remover_fundo_e_salvar(roi_poligonal, caminho_saida)
            print(f"Image saved: {caminho_saida}")

        except Exception as e:
            print(f"Error processing {nome_arquivo}: {str(e)}")